# 🏀 NBA Playoff Game Predictor
**Logistic Regression trained on 10 seasons of team-level playoff data**

---
**Pipeline overview**
1. Load & explore data
2. Feature engineering (net ratings, differentials)
3. Train/test split by season (no data leakage)
4. Logistic regression model
5. Evaluation (accuracy, ROC-AUC, confusion matrix)
6. Feature importance chart


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, classification_report
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

## Step 1 · Load & Explore Data

In [ ]:
df = pd.read_csv('data/playoff_games.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
print(f"Home win rate : {df['home_wins'].mean():.3f}")
print(f"Seasons       : {sorted(df['season'].unique())}")
print(f"Teams         : {sorted(df['home_team'].unique())}")

In [ ]:
# Games per season
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df.groupby('season')['home_wins'].count().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Games per Season')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Game Count')
axes[0].tick_params(axis='x', rotation=45)

# Off/def rating distributions
axes[1].hist(df['home_off_rtg'], bins=30, alpha=0.6, label='Home Off Rtg', color='steelblue')
axes[1].hist(df['away_off_rtg'], bins=30, alpha=0.6, label='Away Off Rtg', color='coral')
axes[1].set_title('Offensive Rating Distribution')
axes[1].set_xlabel('Off Rating (pts / 100 poss)')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/01_data_overview.png', bbox_inches='tight')
plt.show()

## Step 2 · Feature Engineering

Rather than feeding raw home/away stats separately, we compute **differentials** (home minus away). This compresses 12 raw columns into 6 signal-rich features and removes the raw scale, making the logistic regression easier to interpret.

In [ ]:
import os
os.makedirs('plots', exist_ok=True)

df = pd.read_csv('data/playoff_games.csv')

# Net ratings
df['home_net_rtg'] = df['home_off_rtg'] - df['home_def_rtg']
df['away_net_rtg'] = df['away_off_rtg'] - df['away_def_rtg']

# Differentials (home perspective)
df['net_rtg_diff']    = df['home_net_rtg']    - df['away_net_rtg']
df['off_rtg_diff']    = df['home_off_rtg']    - df['away_off_rtg']
df['def_rtg_diff']    = df['home_def_rtg']    - df['away_def_rtg']   # lower is better for defense
df['pace_diff']       = df['home_pace']       - df['away_pace']
df['rest_diff']       = df['home_rest_days']  - df['away_rest_days']
df['form_diff']       = df['home_form_l10']   - df['away_form_l10']

FEATURES = [
    'net_rtg_diff',
    'off_rtg_diff',
    'def_rtg_diff',
    'pace_diff',
    'rest_diff',
    'form_diff',
]
TARGET = 'home_wins'

print('Feature set:', FEATURES)
df[FEATURES + [TARGET]].describe().round(3)

In [ ]:
# Correlation heatmap of features vs target
corr_cols = FEATURES + [TARGET]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, mask=mask,
    linewidths=0.5
)
plt.title('Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('plots/02_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Net rating differential vs win outcome
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat, label in zip(
    axes,
    ['net_rtg_diff', 'rest_diff', 'form_diff'],
    ['Net Rating Diff', 'Rest Day Diff', 'Form (L10) Diff']
):
    for outcome, color, name in [(0, 'coral', 'Away wins'), (1, 'steelblue', 'Home wins')]:
        data = df.loc[df[TARGET] == outcome, feat]
        ax.hist(data, bins=25, alpha=0.6, color=color, label=name)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(label)
    ax.set_xlabel(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plots/03_feature_distributions.png', bbox_inches='tight')
plt.show()

## Step 3 · Train / Test Split by Season

To prevent **data leakage**, we split by season — the model trains on older seasons and tests on the most recent two. This mirrors how the model would actually be used in production.

In [ ]:
seasons_sorted = sorted(df['season'].unique())
print('All seasons:', seasons_sorted)

test_seasons  = seasons_sorted[-2:]   # 2022-23, 2023-24
train_seasons = seasons_sorted[:-2]   # everything older

train_df = df[df['season'].isin(train_seasons)]
test_df  = df[df['season'].isin(test_seasons)]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

print(f'\nTrain seasons : {train_seasons}')
print(f'Test seasons  : {test_seasons}')
print(f'Train size    : {len(X_train)} games')
print(f'Test size     : {len(X_test)} games')
print(f'Train home win%: {y_train.mean():.3f}')
print(f'Test  home win%: {y_test.mean():.3f}')

## Step 4 · Train Model

In [ ]:
# Pipeline: StandardScaler → LogisticRegression
model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ))
])

model.fit(X_train, y_train)
print('Model trained ✓')

## Step 5 · Evaluate

In [ ]:
y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]

acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print('─' * 35)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('─' * 35)
print()
print(classification_report(y_test, y_pred, target_names=['Away wins', 'Home wins']))

In [ ]:
# ── Confusion matrix + ROC curve side by side ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Away wins', 'Home wins'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (Accuracy = {acc:.3f})', fontsize=12)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12)
axes[1].legend(loc='lower right')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('plots/04_evaluation.png', bbox_inches='tight')
plt.show()

## Step 6 · Feature Importance

In [ ]:
# Extract scaled coefficients from the pipeline
lr      = model.named_steps['lr']
coefs   = lr.coef_[0]

feature_labels = {
    'net_rtg_diff' : 'Net Rating Diff',
    'off_rtg_diff' : 'Off Rating Diff',
    'def_rtg_diff' : 'Def Rating Diff',
    'pace_diff'    : 'Pace Diff',
    'rest_diff'    : 'Rest Day Diff',
    'form_diff'    : 'Recent Form Diff (L10)',
}

importance_df = pd.DataFrame({
    'feature' : [feature_labels[f] for f in FEATURES],
    'coef'    : coefs,
    'abs_coef': np.abs(coefs),
}).sort_values('abs_coef', ascending=True)

# Color bars by sign
colors = ['steelblue' if c > 0 else 'coral' for c in importance_df['coef']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importance_df['feature'], importance_df['coef'], color=colors, edgecolor='white')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient (standardized)', fontsize=11)
ax.set_title('Feature Importance — NBA Playoff Predictor', fontsize=13, pad=12)

# Value labels
for bar, val in zip(bars, importance_df['coef']):
    offset = 0.01 if val >= 0 else -0.01
    ax.text(
        val + offset, bar.get_y() + bar.get_height() / 2,
        f'{val:+.3f}', va='center',
        ha='left' if val >= 0 else 'right',
        fontsize=9, color='#333'
    )

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(color='steelblue', label='Favors home team'),
    Patch(color='coral',     label='Favors away team'),
]
ax.legend(handles=legend, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('plots/05_feature_importance.png', bbox_inches='tight')
plt.show()
print(importance_df.to_string(index=False))

In [ ]:
# ── Predicted probability calibration plot ─────────────────────────────────
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10)

plt.figure(figsize=(7, 5))
plt.plot(prob_pred, prob_true, 's-', color='steelblue', label='Logistic Regression')
plt.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig('plots/06_calibration.png', bbox_inches='tight')
plt.show()

## Quick Prediction Demo

Mirroring what the CLI does interactively.

In [ ]:
import json, joblib, os

# Save model artifact for the CLI
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/playoff_predictor.pkl')

# Compute per-team average stats (used by CLI to look up teams)
home_stats = df.groupby('home_team')[[
    'home_off_rtg', 'home_def_rtg', 'home_pace', 'home_form_l10'
]].mean().rename(columns={
    'home_off_rtg': 'off_rtg', 'home_def_rtg': 'def_rtg',
    'home_pace': 'pace', 'home_form_l10': 'form_l10'
})
away_stats = df.groupby('away_team')[[
    'away_off_rtg', 'away_def_rtg', 'away_pace', 'away_form_l10'
]].mean().rename(columns={
    'away_off_rtg': 'off_rtg', 'away_def_rtg': 'def_rtg',
    'away_pace': 'pace', 'away_form_l10': 'form_l10'
})
team_stats = ((home_stats + away_stats) / 2).round(3)
team_stats.to_csv('model/team_averages.csv')

print('Saved → model/playoff_predictor.pkl')
print('Saved → model/team_averages.csv')
print()
print('Available teams:')
print(sorted(team_stats.index.tolist()))

In [ ]:
# In-notebook prediction
def predict_game(home_team, away_team, team_stats_df, trained_model,
                 home_rest=2, away_rest=2):
    """Predict win probability for a playoff matchup."""
    teams_upper = {t.upper(): t for t in team_stats_df.index}
    
    hk = home_team.upper()
    ak = away_team.upper()
    
    if hk not in teams_upper:
        raise ValueError(f"Unknown team: {home_team}. Valid: {sorted(teams_upper.keys())}")
    if ak not in teams_upper:
        raise ValueError(f"Unknown team: {away_team}. Valid: {sorted(teams_upper.keys())}")

    h = team_stats_df.loc[teams_upper[hk]]
    a = team_stats_df.loc[teams_upper[ak]]

    features = pd.DataFrame([{
        'net_rtg_diff': (h['off_rtg'] - h['def_rtg']) - (a['off_rtg'] - a['def_rtg']),
        'off_rtg_diff': h['off_rtg'] - a['off_rtg'],
        'def_rtg_diff': h['def_rtg'] - a['def_rtg'],
        'pace_diff'   : h['pace']    - a['pace'],
        'rest_diff'   : home_rest    - away_rest,
        'form_diff'   : h['form_l10']- a['form_l10'],
    }])

    prob_home = trained_model.predict_proba(features)[0, 1]
    return prob_home


# Demo predictions
matchups = [
    ('BOS', 'MIA', 2, 2),
    ('GSW', 'LAL', 2, 2),
    ('DEN', 'PHX', 3, 1),   # DEN with rest advantage
    ('MIL', 'BKN', 1, 3),   # MIL with fatigue
]

print('\n' + '═'*60)
print(f"  {'MATCHUP':<28} {'HOME WIN%':>10}  {'AWAY WIN%':>10}")
print('═'*60)
for home, away, hr, ar in matchups:
    p = predict_game(home, away, team_stats, model, hr, ar)
    label = f"{home} (home, {hr}d rest) vs {away} ({ar}d rest)"
    print(f"  {label:<38}  {p*100:>6.1f}%  {(1-p)*100:>6.1f}%")
print('═'*60)